In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "4"

import torch
import torch.nn.functional as F
from transformers import T5TokenizerFast, T5EncoderModel

torch.set_grad_enabled(False)

/local2/kunkim/textboost-dev/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch.autograd.grad_mode.set_grad_enabled(mode=False)

In [2]:
model = "black-forest-labs/FLUX.1-dev"

tokenizer = T5TokenizerFast.from_pretrained(model, subfolder="tokenizer_2")
t5_encoder = T5EncoderModel.from_pretrained(model, subfolder="text_encoder_2").cuda()

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers
Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.58it/s]


In [9]:
def tokenize_prompt(tokenizer, prompt, max_sequence_length):
    text_inputs = tokenizer(
        prompt,
        padding="max_length",
        max_length=max_sequence_length,
        truncation=True,
        return_length=False,
        return_overflowing_tokens=False,
        return_tensors="pt",
    )
    text_input_ids = text_inputs.input_ids
    return text_input_ids


def encode_prompt_with_t5(
    text_encoder,
    tokenizer,
    max_sequence_length=512,
    prompt=None,
    num_images_per_prompt=1,
    device=None,
    text_input_ids=None,
):
    prompt = [prompt] if isinstance(prompt, str) else prompt
    batch_size = len(prompt)

    if tokenizer is not None:
        text_inputs = tokenizer(
            prompt,
            padding="max_length",
            max_length=max_sequence_length,
            truncation=True,
            return_length=False,
            return_overflowing_tokens=False,
            return_tensors="pt",
        )
        text_input_ids = text_inputs.input_ids
        print(text_input_ids.shape)
        print(text_input_ids[:, :10])
    else:
        if text_input_ids is None:
            raise ValueError(
                "text_input_ids must be provided when the tokenizer is not specified"
            )

    prompt_embeds = text_encoder(text_input_ids.to(device))[0]

    if hasattr(text_encoder, "module"):
        dtype = text_encoder.module.dtype
    else:
        dtype = text_encoder.dtype
    prompt_embeds = prompt_embeds.to(dtype=dtype, device=device)

    _, seq_len, _ = prompt_embeds.shape

    # duplicate text embeddings and attention mask for each generation per prompt, using mps friendly method
    prompt_embeds = prompt_embeds.repeat(1, num_images_per_prompt, 1)
    prompt_embeds = prompt_embeds.view(batch_size * num_images_per_prompt, seq_len, -1)

    return prompt_embeds, text_input_ids

In [12]:
# 0: padding token
# 1: end of text token
# 1712: cat
# 1782: dog
propmt1 = "photo of a big fluffy brown dog, a cat in the basket"
propmt2 = "b, c, d, a big fluffy brown dog in the basket"
prompts = [propmt1, propmt2]
propmt_embeds, text_input_ids = encode_prompt_with_t5(
    t5_encoder,
    tokenizer,
    max_sequence_length=512,
    prompt=prompts,
    num_images_per_prompt=1,
    device="cuda",
)

dog_idx = torch.where(text_input_ids[0] == 1782)[0][0].item()
dog_idx2 = torch.where(text_input_ids[1] == 1782)[0][0].item()
print(dog_idx, dog_idx2)


cos_sim = F.cosine_similarity(
    propmt_embeds[0:1, dog_idx],
    propmt_embeds[1:2, dog_idx2],
    dim=-1,
)
print(cos_sim)

torch.Size([2, 512])
tensor([[ 1202,    13,     3,     9,   600, 25155,  4216,  1782,     6,     3],
        [    3,   115,     6,     3,    75,     6,     3,    26,     6,     3]])
7 14
tensor([0.7754], device='cuda:0')
